In [3]:
import pandas as pd
import numpy as np

raw_data=pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")
raw_data.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
# To check missing values 
null_count=raw_data.isnull().sum()
print(null_count[null_count>0])
print(null_count)

Series([], dtype: int64)
Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentR

In [7]:
# To check duplicate rows
duplicate_count=raw_data.duplicated().sum()
print(duplicate_count[duplicate_count>0])
print(duplicate_count)

[]
0


In [9]:
# To check constants
nunique=raw_data.nunique()
print(nunique)

Age                           43
Attrition                      2
BusinessTravel                 3
DailyRate                    886
Department                     3
DistanceFromHome              29
Education                      5
EducationField                 6
EmployeeCount                  1
EmployeeNumber              1470
EnvironmentSatisfaction        4
Gender                         2
HourlyRate                    71
JobInvolvement                 4
JobLevel                       5
JobRole                        9
JobSatisfaction                4
MaritalStatus                  3
MonthlyIncome               1349
MonthlyRate                 1427
NumCompaniesWorked            10
Over18                         1
OverTime                       2
PercentSalaryHike             15
PerformanceRating              2
RelationshipSatisfaction       4
StandardHours                  1
StockOptionLevel               4
TotalWorkingYears             40
TrainingTimesLastYear          7
WorkLifeBa

In [5]:
# There are columns with zero statistical variance remove those columns

# Create our clean working copy to preserve data lineage
cleaned_data = raw_data.copy()

# List the 4 zero-variance columns you found
columns_to_drop = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']

# Drop them safely from the operational dataframe
cleaned_data.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# Verify the new structural shape
print(f"Original Dataset Columns: {raw_data.shape[1]}")
print(f"Cleaned Dataset Columns:  {cleaned_data.shape[1]}")


Original Dataset Columns: 35
Cleaned Dataset Columns:  31


In [ ]:
# To check for data types
print(raw_data.dtypes)

In [16]:
# To check outliers 
raw_data[['Age','MonthlyIncome']].describe()

,Age,MonthlyIncome
count,1470.000000,1470.000000
mean,36.923810,6502.931293
std,9.135373,4707.956783
min,18.000000,1009.000000
25%,30.000000,2911.000000
50%,36.000000,4919.000000
75%,43.000000,8379.000000
max,60.000000,19999.000000


In [20]:
# We can see a sudden jump in the numbers around monthly income from 75% to the maximum mark.

# 1. Sort the data by Monthly Income from highest to lowest
highest_salaries = raw_data[['JobRole', 'MonthlyIncome']].sort_values(by='MonthlyIncome', ascending=False)

# 2. Print out the top 10 highest earners
print(highest_salaries.head(10))

                JobRole  MonthlyIncome
190             Manager          19999
746   Research Director          19973
851             Manager          19943
165             Manager          19926
568             Manager          19859
918             Manager          19847
749             Manager          19845
1242            Manager          19833
898   Research Director          19740
956             Manager          19717


In [21]:
# Lets check logical contradiction
# 1. Count rows where Total Working Years is physically impossible compared to Age
# (Assuming a person must be at least 14-16 years old to start counting professional years)
age_contradictions = cleaned_data[cleaned_data['Age'] - cleaned_data['TotalWorkingYears'] < 14]

# 2. Count rows where tenure at this company is longer than their entire career
tenure_contradictions = cleaned_data[cleaned_data['TotalWorkingYears'] < cleaned_data['YearsAtCompany']]

# 3. Print the diagnostic results
print(len(age_contradictions))
print(len(tenure_contradictions))

0
0


In [22]:
# To check for categorical inconsistency
# Print the unique text categories for our main dashboard columns
print(cleaned_data['Department'].unique())
print(cleaned_data['BusinessTravel'].unique())
print(cleaned_data['EducationField'].unique())
print(cleaned_data['JobRole'].unique())
print(cleaned_data['MaritalStatus'].unique())

['Sales' 'Research & Development' 'Human Resources']
['Travel_Rarely' 'Travel_Frequently' 'Non-Travel']
['Life Sciences' 'Other' 'Medical' 'Marketing' 'Technical Degree'
 'Human Resources']
['Sales Executive' 'Research Scientist' 'Laboratory Technician'
 'Manufacturing Director' 'Healthcare Representative' 'Manager'
 'Sales Representative' 'Research Director' 'Human Resources']
['Single' 'Married' 'Divorced']


In [23]:
# To check data leakage target variable

# Print out all 31 remaining columns so we can visually inspect them for leakage
print(cleaned_data.columns.tolist())

['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [24]:
cleaned_data.to_csv('HR_Employee_Attrition_Cleaned.csv', index=False)
print("The dataset has been cleaned successfully!")

The dataset has been cleaned successfully!


In [6]:
# Calculation 1: Data Schema & Feature Identification
# Separate remaining columns by data type
numerical_cols = cleaned_data.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = cleaned_data.select_dtypes(include=['object']).columns.tolist()

print(f"📈 Numerical Columns ({len(numerical_cols)}):\n{numerical_cols}\n")
print(f"🔤 Categorical Columns ({len(categorical_cols)}):\n{categorical_cols}")

📈 Numerical Columns (23):
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

🔤 Categorical Columns (8):
['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']


In [7]:
# Calculation 2: Baseline HR Key Performance Indicators

# Core HR Metrics
total_employees = len(cleaned_data)
total_attrition = (cleaned_data['Attrition'] == 'Yes').sum()
baseline_attrition_rate = (cleaned_data['Attrition'] == 'Yes').mean() * 100
avg_monthly_income = cleaned_data['MonthlyIncome'].mean()

print(f"1. Total Employee Roster: {total_employees}")
print(f"2. Total Attrition Count (Exits): {total_attrition}")
print(f"3. Baseline Company Attrition Rate: {baseline_attrition_rate:.2f}%")
print(f"4. Average Monthly Workforce Salary: ${avg_monthly_income:.2f}")

1. Total Employee Roster: 1470
2. Total Attrition Count (Exits): 237
3. Baseline Company Attrition Rate: 16.12%
4. Average Monthly Workforce Salary: $6502.93


In [8]:
# Calculation 3: Attrition Across Demographic & Operational Segments

# A. Attrition Rate by Department
print("--- Attrition % by Department ---")
dept_calc = cleaned_data.groupby('Department')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
print(dept_calc.round(2))

# B. Attrition Rate by Job Role
print("\n--- Attrition % by Job Role ---")
role_calc = cleaned_data.groupby('JobRole')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=False)
print(role_calc.round(2))

# C. Attrition Rate by Business Travel Frequency
print("\n--- Attrition % by Travel Rules ---")
travel_calc = cleaned_data.groupby('BusinessTravel')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
print(travel_calc.round(2))

# D. Attrition Rate by Marital Status
print("\n--- Attrition % by Marital Status ---")
marital_calc = cleaned_data.groupby('MaritalStatus')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
print(marital_calc.round(2))

--- Attrition % by Department ---
Department
Human Resources           19.05
Research & Development    13.84
Sales                     20.63
Name: Attrition, dtype: float64

--- Attrition % by Job Role ---
JobRole
Sales Representative         39.76
Laboratory Technician        23.94
Human Resources              23.08
Sales Executive              17.48
Research Scientist           16.10
Manufacturing Director        6.90
Healthcare Representative     6.87
Manager                       4.90
Research Director             2.50
Name: Attrition, dtype: float64

--- Attrition % by Travel Rules ---
BusinessTravel
Non-Travel            8.00
Travel_Frequently    24.91
Travel_Rarely        14.96
Name: Attrition, dtype: float64

--- Attrition % by Marital Status ---
MaritalStatus
Divorced    10.09
Married     12.48
Single      25.53
Name: Attrition, dtype: float64


In [10]:
# Calculation 4: Behavioral, Sentiment, & Compensation Triggers  

# A. The Overtime Multiplier
print("--- Overtime Impact on Exit Rates ---")
ot_calc = cleaned_data.groupby('OverTime')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
print(ot_calc.round(2))

# B. Salary Disparity (Compare average income of those who stayed vs left)
print("\n--- Average Monthly Income Disparity ---")
income_calc = cleaned_data.groupby('Attrition')['MonthlyIncome'].mean()
print(income_calc.round(2))

# C. Job Satisfaction Score Impact (1 = Low, 4 = High)
print("\n--- Attrition % by Job Satisfaction Level ---")
sat_calc = cleaned_data.groupby('JobSatisfaction')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
print(sat_calc.round(2))

# D. Work-Life Balance Score Impact (1 = Bad, 4 = Best)
print("\n--- Attrition % by Work-Life Balance Level ---")
wlb_calc = cleaned_data.groupby('WorkLifeBalance')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)
print(wlb_calc.round(2))

--- Overtime Impact on Exit Rates ---
OverTime
No     10.44
Yes    30.53
Name: Attrition, dtype: float64

--- Average Monthly Income Disparity ---
Attrition
No     6832.74
Yes    4787.09
Name: MonthlyIncome, dtype: float64

--- Attrition % by Job Satisfaction Level ---
JobSatisfaction
1    22.84
2    16.43
3    16.52
4    11.33
Name: Attrition, dtype: float64

--- Attrition % by Work-Life Balance Level ---
WorkLifeBalance
1    31.25
2    16.86
3    14.22
4    17.65
Name: Attrition, dtype: float64


In [11]:
# The Missing Piece: Automated Data Dictionary & Completeness Check (Guideline 1)

import pandas as pd

# Generate structural metadata for the data dictionary
data_dictionary_summary = pd.DataFrame({
    'Data Type': cleaned_data.dtypes,
    'Non-Null Count': cleaned_data.notnull().sum(),
    'Unique Values Count': cleaned_data.nunique()
})

print("--- 📚 DATA DICTIONARY STRUCTURAL BASELINE ---")
print(data_dictionary_summary)

--- 📚 DATA DICTIONARY STRUCTURAL BASELINE ---
                         Data Type  Non-Null Count  Unique Values Count
Age                          int64            1470                   43
Attrition                   object            1470                    2
BusinessTravel              object            1470                    3
DailyRate                    int64            1470                  886
Department                  object            1470                    3
DistanceFromHome             int64            1470                   29
Education                    int64            1470                    5
EducationField              object            1470                    6
EnvironmentSatisfaction      int64            1470                    4
Gender                      object            1470                    2
HourlyRate                   int64            1470                   71
JobInvolvement               int64            1470                    4
JobLevel          